In [1]:
#some imports
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, FunctionTransformer, SplineTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="'penalty' was deprecated in version 1.8",
    category=FutureWarning
)

## Feature engineering

In [3]:
binary_columns = [
    "diabetes",
    "hypertension",
    "chronic_kidney_disease",
    "readmitted_30d"
]

int_columns = ["age", "systolic_bp", "diastolic_bp", "heart_rate", "glucose_mg_dl", "sodium_mmol_l", "respiratory_rate", "spo2"]

In [4]:
def feature_engineering(data):
    data = data.copy()

    #feature engineering without dropping
    data["mean_arterial_pressure"] = (
        data["systolic_bp"] + 2 * data["diastolic_bp"]
    ) / 3

    data["bmi_age_interaction"] = (
        data["bmi"] * data["age"]
    )

    data["pulse_pressure"] = data["systolic_bp"] - data["diastolic_bp"]

    data["log_length"] = np.log1p(data["length_of_stay_days"])

    data["log_creatinine"] = np.log(data["creatinine_mg_dl"])

    # yes no columns to 0 and 1
    for column in binary_columns:
        data[column] = data[column].map({"No": 0, "Yes": 1})

    # convert int and NaN to string
    def to_str(x):
        return x.astype("Int64").astype(str).where(x.notna(), "nan")

    #dictionary for new features
    new = {}

    # last digit of integer columns ("digit preference" of each LLM)
    for c in int_columns:
            new[c + "_ld"] = to_str(data[c] % 10)


    # tenths digit of 1-decimal columns, hundredths of creatinine, respiratory rate parity
    for c in ["bmi", "temperature_c", "hemoglobin_g_dl", "length_of_stay_days"]:
            new[c + "_ld"] = to_str((data[c] * 10).round() % 10)
    new["creat_dec2"] = to_str((data["creatinine_mg_dl"] * 100).round() % 10)
    new["rr_parity"] = to_str(data["respiratory_rate"] % 2)

    # exact value as a category (each LLM has its own "vocabulary" of values)
    for c in ["temperature_c", "bmi", "hemoglobin_g_dl", "length_of_stay_days",
            "creatinine_mg_dl"]:
        new[c + "_ex"] = data[c].round(2).astype(str).where(data[c].notna(), "nan")

    for c in ["num_medications", "prior_visits_12m"] + int_columns:
        new[c + "_ex"] = to_str(data[c])


    return pd.concat([data, pd.DataFrame(new, index=data.index)], axis=1)

vaig a intentar aplicar lo de la l1 per veure quines son les millors variables

In [ ]:
#   0. carreguem les dades 
train = pd.read_csv('./train.csv')

X = train.drop(columns=["id","label"])
y = train["label"]

# 1. apliquem feature engineering

X = feature_engineering(X)

# 2. dividim entre categorical i numerical features. 

cat_cols = [c for c in X if X[c].dtype == object or str(X[c].dtype) in ("str", "string")]
num_cols = [c for c in X if c not in cat_cols]

num_robust = ["length_of_stay_days", "creatinine_mg_dl", "prior_visits_12m"] # Exemples amb possibles outliers
num_standard = [c for c in num_cols if c not in num_robust]

#  3. Decidim preprocesor and pipelines

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_robust),

    # rare levels are grouped together -> keeps penalty=None from overfitting
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=10, sparse_output=False), cat_cols),
])

# 4. Preparem les dades ja transformades pel preprocessor actual
X_processed = preprocessor.fit_transform(X)

# 5. Obtenim els noms de totes les columnes després de l'OneHotEncoder i el Pipeline
cat_encoder = preprocessor.named_transformers_["cat"]
# Obtenim els noms de les categories codificades
cat_out_cols = cat_encoder.get_feature_names_out(cat_cols)
all_feature_names = list(num_cols) + list(cat_out_cols)

# 3. Entrenem un model L1 ràpid per veure quins coeficients no són zero
l1_model = LogisticRegression(
    penalty="l1", 
    l1_ratio=1.0,       
    solver="saga", 
    C=0.1, 
    max_iter=5000, 
    random_state=42
)
l1_model.fit(X_processed, y)

# 4. Creem un DataFrame per veure el pes de cada característica
# Nota: Si el teu problema és multiclasse, l1_model.coef_ té una fila per classe. 
# Podem fer la mitjana del valor absolut dels coeficients per veure la importància global.
importances = np.abs(l1_model.coef_).mean(axis=0)

feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

# 5. Mostrem les característiques més importants i les descartades (pes 0)
print("--- CARACTERÍSTIQUES MÉS IMPORTANTS ---")
print(feature_importance_df.head(20))

print("\n--- CARACTERÍSTIQUES DESCARTADES (PES ZERO) ---")
zero_features = feature_importance_df[feature_importance_df["importance"] == 0]
print(f"Total de característiques amb pes zero: {len(zero_features)} de {len(feature_importance_df)}")
print(zero_features.tail(20))

--- CARACTERÍSTIQUES MÉS IMPORTANTS ---
                       feature  importance
0                          age    0.832853
162   length_of_stay_days_ld_0    0.638060
18              readmitted_30d    0.527048
58            systolic_bp_ld_0    0.511810
31    encounter_type_Emergency    0.350214
372      prior_visits_12m_ex_0    0.334483
7                         spo2    0.329220
20         bmi_age_interaction    0.315405
17         length_of_stay_days    0.313685
68           diastolic_bp_ld_0    0.312628
32   encounter_type_Outpatient    0.306710
8                glucose_mg_dl    0.302835
9             creatinine_mg_dl    0.298109
378      prior_visits_12m_ex_4    0.282308
16            prior_visits_12m    0.277430
12                    diabetes    0.266749
648     respiratory_rate_ex_17    0.242209
131                   bmi_ld_1    0.237275
5             respiratory_rate    0.232267
6                temperature_c    0.231065

--- CARACTERÍSTIQUES DESCARTADES (PES ZERO) ---
Total de

In [41]:
# 1. Creem una columna nova al DataFrame d'importàncies que extregui el nom base de la variable
# (eliminant sufixos com _ld_X, _ex_X, etc., basant-nos en els patrons coneguts)
import re

def get_base_feature_name(feat):
    # Elimina sufixos tipus _ld_..., _ex_..., o categories codificades d'OneHotEncoder
    # Per exemple: "sodium_mmol_l_ld_9113" -> "sodium_mmol_l"
    for col in int_columns + binary_columns + ["sex","creat_dec2","num_medications", "rr_parity","prior_visits_12m", "discharge_destination", "bmi", "temperature_c", "hemoglobin_g_dl", "length_of_stay_days", "creatinine_mg_dl", "systolic_bp", "diastolic_bp", "heart_rate", "glucose_mg_dl", "sodium_mmol_l", "respiratory_rate", "spo2", "encounter_type", "smoking_status", "primary_diagnosis"]:
        if feat.startswith(col):
            return col
    return feat

feature_importance_df["base_feature"] = feature_importance_df["feature"].apply(get_base_feature_name)

# 2. Agrupem per la variable base fent la mitjana (o la suma) de la seva importància
grouped_importances = feature_importance_df.groupby("base_feature")["importance"].mean().reset_index()
grouped_importances = grouped_importances.sort_values(by="importance", ascending=False)

print("--- IMPORTÀNCIA GLOBAL PER VARIABLE BASE ---")
print(grouped_importances.head(50))

--- IMPORTÀNCIA GLOBAL PER VARIABLE BASE ---
              base_feature  importance
21          readmitted_30d    0.527048
5                 diabetes    0.266749
8           encounter_type    0.206685
23               rr_parity    0.197279
2   chronic_kidney_disease    0.083051
16  mean_arterial_pressure    0.074702
19        prior_visits_12m    0.070284
12            hypertension    0.052175
24                     sex    0.050205
13     length_of_stay_days    0.042133
22        respiratory_rate    0.037087
27                    spo2    0.037026
29           temperature_c    0.033895
3               creat_dec2    0.022361
0                      age    0.020763
25          smoking_status    0.015837
14          log_creatinine    0.015172
4         creatinine_mg_dl    0.014911
17         num_medications    0.013761
18       primary_diagnosis    0.012789
26           sodium_mmol_l    0.012289
1                      bmi    0.012081
28             systolic_bp    0.011209
9            glucos

eliminem discharge_destination ja que te 0

## cross validation amb train

In [87]:
train = pd.read_csv('./train.csv')

In [88]:
X = train.drop(columns=["id","label","discharge_destination"])
y = train["label"]

In [89]:
X = feature_engineering(X)

In [90]:
# categorical and numerical features. no les llistem pq ara hi ha moltes

cat_cols = [c for c in X if X[c].dtype == object or str(X[c].dtype) in ("str", "string")]
num_cols = [c for c in X if c not in cat_cols]

In [91]:
#preprocesor and pipelines
from sklearn.impute import KNNImputer

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", KNNImputer(n_neighbors=8)),
                      ("sc", StandardScaler())]), num_cols),

    # rare levels are grouped together -> keeps penalty=None from overfitting
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=10, sparse_output=False), cat_cols),
])

In [92]:
#model
model = Pipeline([("pre", preprocessor),
                  ("clf", LogisticRegression(penalty=None, max_iter=10000))])

In [93]:
#cross validation
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)
scores = cross_val_score(
    model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy for each fold:", scores.round(4))
print("Mean validation accuracy:", round(scores.mean(), 4))
print("Variation between folds:", round(scores.std(), 4))

Accuracy for each fold: [0.9333 0.9229 0.925  0.9312 0.8958 0.9125 0.925  0.925  0.9229 0.9083
 0.9271 0.9188 0.9375 0.9125 0.9208 0.9167 0.9188 0.9146 0.9229 0.9125
 0.9104 0.9229 0.9333 0.925  0.9167]
Mean validation accuracy: 0.9205
Variation between folds: 0.0089


**El de l'azu:**

Accuracy for each fold: [0.9458 0.9167 0.9333 0.9208 0.9    0.9104 0.9312 0.9188 0.9208 0.8979
 0.9146 0.9125 0.9229 0.9104 0.9188 0.9229 0.925  0.9125 0.9208 0.9104
 0.9021 0.9208 0.9333 0.9292 0.925 ]

Mean validation accuracy: 0.9191

Variation between folds: 0.0108

**Sense "discharge_destination": -> utilitzant l1**

Accuracy for each fold: [0.9417 0.9229 0.9312 0.9208 0.9    0.9146 0.9292 0.9271 0.9125 0.9062
 0.9188 0.9167 0.9271 0.9042 0.9167 0.9229 0.9229 0.9146 0.925  0.9146
 0.9083 0.9208 0.9312 0.9312 0.9292]

Mean validation accuracy: 0.9204

Variation between folds: 0.0096

**canvi de StandardScaler a RobustScaler**: ( ha baixat )


**canvi a KKN neighbours**: 

Accuracy for each fold: [0.9396 0.925  0.9229 0.9292 0.9021 0.9062 0.925  0.925  0.9208 0.9125
 0.9229 0.9188 0.9375 0.9125 0.9229 0.9104 0.9188 0.9104 0.9271 0.9062
 0.9146 0.9271 0.9292 0.9271 0.9188]

Mean validation accuracy: 0.9205

Variation between folds: 0.0092

## to submit

In [110]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

In [111]:
X_train = train.drop(columns=["id","label","discharge_destination"])
y_train = train["label"]

test_ids = test["id"]
X_test = test.drop(columns=["id","discharge_destination"])

apliquem la funció definida adalt

In [112]:
X_train = feature_engineering(X_train)
X_test = feature_engineering(X_test)

In [113]:
cat_cols = [c for c in X_train if X_train[c].dtype == object or str(X_train[c].dtype) in ("str", "string")]
num_cols = [c for c in X_train if c not in cat_cols]

In [ ]:
#preprocesor and pipelines

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", KNNImputer(n_neighbors=10)),
                      ("sc", StandardScaler())]), num_cols),

    # rare levels are grouped together -> keeps penalty=None from overfitting
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=10, sparse_output=False), cat_cols),
])

In [115]:
model = Pipeline([("pre", preprocessor),
                  ("clf", LogisticRegression(penalty=None, max_iter=10000))])

In [116]:
model.fit(X_train, y_train)

predictions = model.predict(X_test)

# Create the submission DataFrame
submission = pd.DataFrame({
    "id": test_ids,
    "label": predictions
})

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("Train accuracy:", accuracy_score(y_train, model.predict(X_train)))
print(submission.head())
print("Submission shape:", submission.shape)

Train accuracy: 1.0
                                 id    label
0  0102f98c5fd7ed84754c58fe0d7f013f  Model_D
1  e2fa3e497f5bdb8c79748c7c0611fad0  Model_B
2  818a6ec782d097abc1ce67c42299d06a  Model_D
3  54e46a096cb47b5499a986477be83061  Model_B
4  5da9e7a746719fc3950c6e579c82bfa1  Model_C
Submission shape: (1600, 2)


In [117]:
assert list(X_train.columns) == list(X_test.columns)
print(submission["label"].value_counts())  

label
Model_B    422
Model_D    399
Model_C    396
Model_A    383
Name: count, dtype: int64


Each LLM writes numbers in its own style so we added features to find the patterns in their styles.

Last digit and decimal digits: each model has its own habits, like preferring round numbers or writing a certain number of decimals. A numeric column can’t capture this, so we turn the digit into a category.

Exact value as a category: each model uses its own set of typical values. Logistic regression with a numeric column only learns a straight trend, so one-hot encoding the exact value lets it learn which values are typical of which model. We group rare values to avoid overfitting.

We removed the missing-value indicators as they did not improve cross-validated accuracy, and the outlier clipping altered the exact values we now use as features.